In [ ]:

import os
from langchain.chat_models import init_chat_model

# Set your NEW API key here after regenerating it
os.environ["OPENAI_API_KEY"] = "Enter your new API key here"

# Initialize the model
model = init_chat_model("gpt-4o-mini", temperature=0.1)

# Test it
response = model.invoke("Hello World!")
display(response.content)

'Hello! How can I assist you today?'

In [2]:
# # Test it
# response = model.invoke("Say hello!")
# print(response.content)
# Configure the database 
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Climate.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM climate_data LIMIT 5;")}')


#Add tools for database interaction
from langchain_community.agent_toolkits import SQLDatabaseToolkit
toolkit = SQLDatabaseToolkit(db=db, llm=model)
tools = toolkit.get_tools()

print(tools)

Dialect: sqlite
Available tables: ['climate_data']
Sample output: [('2020-01-01', 'Germany', 28.29, 31.08, 212.63, 11348.75, 14.42, 76.39, 51.22, 83.93), ('2020-01-02', 'Germany', 28.38, 37.94, 606.05, 4166.64, 5.63, 86.26, 78.27, 110.4), ('2020-01-03', 'Germany', 28.74, 57.67, 268.72, 4503.8, 14.2, 75.92, 48.96, 173.58), ('2020-01-04', 'Germany', 26.66, 51.34, 167.32, 3259.13, 13.84, 63.15, 97.42, 89.13), ('2020-01-05', 'Germany', 26.81, 65.38, 393.89, 7023.72, 6.93, 76.02, 81.89, 40.6)]
[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x10897fa40>), InfoSQLDatabaseTool(descript

In [3]:
system_prompt = """
You are an agent designed to interact with a SQL database about
global climate and energy data spanning 20 countries from 2020 to 2024.

Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.

Additional instructions specific to this database:
- The database contains daily climate and energy records for 20 countries
  including USA, China, India, Germany, Brazil and others.
- Key columns are: date, country, avg_temperature, humidity, co2_emission,
  energy_consumption, renewable_share, urban_population,
  industrial_activity_index, energy_price.
- When comparing countries always use AVG() to get meaningful comparisons
  since data is daily.
- When asked about trends always GROUP BY year or month using
  strftime('%Y', date) or strftime('%Y-%m', date).
- When asked about renewable energy use the renewable_share column.
- When asked about pollution or emissions use the co2_emission column.
- Always present numbers rounded to 2 decimal places using ROUND().
- When ranking countries always use ORDER BY and AVG() together.
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
)

In [6]:
question = "What is the average energy consumption in Germany in 2023?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the average energy consumption in Germany in 2023?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_6U70q09XMGG30r43Vlre5I2g)
 Call ID: call_6U70q09XMGG30r43Vlre5I2g
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

climate_data
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_7ByLH81O12zcXeLX3ok50OdN)
 Call ID: call_7ByLH81O12zcXeLX3ok50OdN
  Args:
    table_names: climate_data
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE climate_data (
	date TEXT, 
	country TEXT, 
	avg_temperature REAL, 
	humidity REAL, 
	co2_emission REAL, 
	energy_consumption REAL, 
	renewable_share REAL, 
	urban_population REAL, 
	industrial_activity_index R